# DSPyUI: A Gradio interface for DSPy

This notebook accompanies Chapter 10 §10.3 of *Context Engineering with DSPy*: **Building a Gradio User Interface (DSPyUI)**. The premise is simple: if signatures define what goes in and what comes out, modules define how to process it, and optimizers define how to improve it, then all of that can be driven through a web form. No Python required for the end user — they define a task, upload a CSV, pick an optimizer, hit compile, and test the result. The full open-source app lives at <https://github.com/hammer-mt/DSPyUI>; this notebook walks through the core building blocks.

**Required environment variables** (loaded from `.env`):

- `OPENAI_API_KEY` — plus any provider keys for models you list in   `MODEL_OPTIONS` below (`ANTHROPIC_API_KEY`, `GEMINI_API_KEY`).

**Service dependencies**: none. Gradio runs in-process; launching the app exposes a local web server.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
%pip install gradio -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5.6-luna'))

## 10.3.1 Signatures

The signature builder is the heart of DSPyUI. Users specify input field names and descriptions, output field names and descriptions, and task instructions — and that form maps directly to `dspy.Signature` with `InputField` and `OutputField`. The engine below uses `make_signature`, DSPy's programmatic way to create signatures at runtime, instead of the class-based syntax you use in code.

In the Gradio UI, this looks like a form with expandable sections. Users fill in their task description (*"Rate how funny this joke is on a scale of 1 to 10"*), define their input fields (`joke_topic`), define their output fields (`funny_rating`), and the signature is constructed behind the scenes.

In [ ]:
from dspy.signatures.signature import make_signature

def create_custom_signature(input_fields, output_fields, instructions,
                            input_descs, output_descs):
    fields = {}
    for i, name in enumerate(input_fields):
        desc = input_descs[i] if i < len(input_descs) and input_descs[i] else None
        fields[name] = (str, dspy.InputField(desc=desc))
    for i, name in enumerate(output_fields):
        desc = output_descs[i] if i < len(output_descs) and output_descs[i] else None
        fields[name] = (str, dspy.OutputField(desc=desc))

    return make_signature(fields, instructions=instructions)

## 10.3.2 Modules

Module selection is a dropdown: `Predict` or `ChainOfThought`. Each maps to the DSPy module from Chapter 7. The model dropdowns let the user pick a **student model** (cheap, runs in production) and a **teacher model** (strong, generates demonstrations or reflection during optimization). This example defaults to teacher `openai/gpt-5.6-sol` and student `openai/gpt-5.6-luna`.

In [ ]:
MODEL_OPTIONS = [
    "openai/gpt-5.6-luna",
    "openai/gpt-5.6-sol",
    "anthropic/claude-opus-4.8",
    "gemini/gemini-3.5-flash",
]

In [ ]:
def create_module(module_name, signature):
    if module_name == "Predict":
        return dspy.Predict(signature)
    if module_name == "ChainOfThought":
        return dspy.ChainOfThought(signature)
    raise ValueError(f"Unsupported module: {module_name}")

Both `dspy.Predict` and `dspy.ChainOfThought` are already `dspy.Module` subclasses, so they work directly with `compile()`, `save()`, and `load()`. No wrapper classes needed.

## 10.3.3 Datasets

Users upload a CSV with columns matching their signature fields. The interface maps columns to fields and splits the data 80/20 into train and test sets. For users who don't have a CSV yet, DSPyUI includes a manual entry mode using `gr.Dataframe` where you type examples directly into a table. This connects back to Chapter 4's dataset strategies — everything you learned about collecting examples, whether from error analysis, domain experts, or synthetic bootstrapping, feeds into this upload step.

DSPyUI ships with three demo CSVs — rating jokes, telling jokes, and rewriting jokes — so new users can see the full workflow without bringing their own data.

In [ ]:
import pandas as pd

def load_and_split(file_path, input_fields, output_fields):
    df = pd.read_csv(file_path)
    fields = input_fields + output_fields
    missing = [field for field in fields if field not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing columns: {', '.join(missing)}")
    if len(df) < 2:
        raise ValueError("Upload at least two examples so both splits are non-empty")

    dataset = [
        dspy.Example(
            **{field: str(row[field]) for field in fields}
        ).with_inputs(*input_fields)
        for _, row in df.iterrows()
    ]

    split = min(max(1, int(0.8 * len(dataset))), len(dataset) - 1)
    return dataset[:split], dataset[split:]

## 10.3.4 Optimization

The compact UI runs a baseline evaluation, compiles against the training set, and evaluates the optimized program. `EvaluationResult.score` uses a **0–100 percentage scale**, so a result is displayed as *"Baseline: 45.00%, Optimized: 78.00%"* rather than 0.45 and 0.78.

| Optimizer label | DSPy implementation | Best for |
| :--- | :--- | :--- |
| `BootstrapFewShot` | `dspy.BootstrapFewShot` | Small datasets (~10 examples), quick iteration |
| `BootstrapRS` | `dspy.BootstrapFewShotWithRandomSearch` | Medium datasets (50+), need reliable improvement |
| `MIPROv2` | `dspy.MIPROv2` | Large datasets (100+), production optimization |
| `GEPA` | `dspy.GEPA` | Creative/subjective tasks, instruction tuning |

For GEPA, supply exactly one budget argument and a reflection LM. The five-argument metric may return a numeric score; subjective tasks should return `dspy.Prediction(score=..., feedback=...)` so GEPA has textual feedback to reflect on.

In [ ]:
from pathlib import Path

def compile_program(module, trainset, devset, optimizer_name, metric,
                    teacher_model="openai/gpt-5.6-sol",
                    save_path="programs/latest.json"):
    baseline_eval = dspy.Evaluate(metric=metric, devset=devset, num_threads=1)
    baseline_score = baseline_eval(module).score

    teacher = module.deepcopy()
    teacher.set_lm(dspy.LM(teacher_model))

    if optimizer_name == "BootstrapFewShot":
        optimizer = dspy.BootstrapFewShot(metric=metric)
        compiled = optimizer.compile(module, teacher=teacher, trainset=trainset)
    elif optimizer_name == "BootstrapRS":
        optimizer = dspy.BootstrapFewShotWithRandomSearch(metric=metric)
        compiled = optimizer.compile(
            module, teacher=teacher, trainset=trainset, valset=devset,
        )
    elif optimizer_name == "MIPROv2":
        optimizer = dspy.MIPROv2(metric=metric, auto="light")
        compiled = optimizer.compile(
            module, teacher=teacher, trainset=trainset, valset=devset,
        )
    elif optimizer_name == "GEPA":
        optimizer = dspy.GEPA(
            metric=metric,
            auto="light",
            reflection_lm=dspy.LM(teacher_model, temperature=1.0),
        )
        compiled = optimizer.compile(module, trainset=trainset, valset=devset)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    final_eval = dspy.Evaluate(metric=metric, devset=devset, num_threads=1)
    final_score = final_eval(compiled).score

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    compiled.save(save_path)

    return baseline_score, final_score, compiled

## 10.3.5 Inference

The final tab loads a saved program and lets users run inference through a form: select a program from the saved-programs directory, enter input values, see the output. This is the payoff — a domain expert who defined a signature, uploaded examples, and ran an optimizer can now interact with the optimized program without touching Python.

The interface shows both the structured output *and* the full prompt that was sent to the LM. That transparency is deliberate — users can see exactly what instructions and examples the optimizer produced, building trust in the system.

In [ ]:
def run_inference(program_path, signature, module_name, input_data):
    # Recreate the module and load saved state
    module = create_module(module_name, signature)
    module.load(program_path)

    # Run inference
    result = module(**input_data)

    # Also extract the prompt for transparency
    messages = dspy.settings.lm.history[-1]["messages"]

    return result, messages

## Putting it together

DSPyUI can separate signature, module, dataset, optimization, and inference into individual screens. The minimal app below keeps the same workflow in one runnable Gradio form and includes both student and teacher model choices. It implements four optimizer options and saves a state-only program that the inference helper above can reload. Launch it with `demo.launch()` to open the local app.

In [ ]:
import gradio as gr

def end_to_end(instructions, input_fields_csv, output_fields_csv,
               module_name, student_model, teacher_model,
               optimizer_name, dataset_path):
    input_fields = [s.strip() for s in input_fields_csv.split(",") if s.strip()]
    output_fields = [s.strip() for s in output_fields_csv.split(",") if s.strip()]
    if not input_fields or not output_fields:
        raise gr.Error("Provide at least one input and one output field")

    signature = create_custom_signature(
        input_fields=input_fields,
        output_fields=output_fields,
        instructions=instructions,
        input_descs=[""] * len(input_fields),
        output_descs=[""] * len(output_fields),
    )

    dspy.configure(lm=dspy.LM(student_model))
    module = create_module(module_name, signature)
    trainset, devset = load_and_split(dataset_path, input_fields, output_fields)

    def exact_match(example, pred, trace=None, pred_name=None, pred_trace=None):
        return float(all(
            str(getattr(pred, field, "")).strip()
            == str(getattr(example, field, "")).strip()
            for field in output_fields
        ))

    baseline_score, final_score, _ = compile_program(
        module,
        trainset,
        devset,
        optimizer_name,
        exact_match,
        teacher_model=teacher_model,
    )
    return f"Baseline: {baseline_score:.2f}%, Optimized: {final_score:.2f}%"

with gr.Blocks() as demo:
    gr.Markdown("# DSPyUI — minimal end-to-end workflow")
    instructions = gr.Textbox(label="Task instructions")
    input_fields_csv = gr.Textbox(label="Input fields (comma-separated)")
    output_fields_csv = gr.Textbox(label="Output fields (comma-separated)")
    module_name = gr.Dropdown(
        ["Predict", "ChainOfThought"], value="ChainOfThought", label="Module",
    )
    student_model = gr.Dropdown(
        MODEL_OPTIONS, value="openai/gpt-5.6-luna", label="Student model",
    )
    teacher_model = gr.Dropdown(
        MODEL_OPTIONS, value="openai/gpt-5.6-sol", label="Teacher model",
    )
    optimizer_name = gr.Dropdown(
        ["BootstrapFewShot", "BootstrapRS", "MIPROv2", "GEPA"],
        value="BootstrapFewShot",
        label="Optimizer",
    )
    dataset_path = gr.File(label="Dataset CSV", type="filepath")
    compile_btn = gr.Button("Compile")
    result = gr.Markdown()
    compile_btn.click(
        end_to_end,
        inputs=[
            instructions, input_fields_csv, output_fields_csv, module_name,
            student_model, teacher_model, optimizer_name, dataset_path,
        ],
        outputs=result,
    )

# demo.launch()